# GEMM-KKT Experiment Interpretation Guide

This notebook explains:
- what each solver/method is doing,
- how experiments are configured, and
- how to interpret the generated summary tables and plots.


## Methods in this repo

- **`gemm_ipm_ns` (`ns_ipm`)**: Primal-dual IPM with factorization-free NS inverse + refinement (`kkt_mode=ns_only`).
- **`gemm_ipm_robust` (`ns_ipm_rb`)**: Same IPM, but can switch to NS-preconditioned Krylov when NS contraction is poor (`kkt_mode=robust`).
- **`gemm_splitting_qp` (`split_qp`)**: ADMM/OSQP-style splitting method with CG+NS or NS linear solve.
- **`lp_first_order_gpu` (`lp_pdhg_gpu`)**: first-order LP solver (PDHG/PDLP-style).
- **`scipy_trust_constr_cpu` (`scipy_trust_cpu`)**: CPU baseline via SciPy trust-constr (single-instance wrapper).
- **`osqp_cpu` / `highs_cpu`**: optional CPU baselines when installed.


## Experimental setup (high-level)

Main QP family is dense batched parametric QP:
\n\[\n\min_x \; \tfrac12 x^T P x + q^T x \quad \text{s.t.}\quad l \le Ax \le u\n\]

Typical presets:
- `ns_favor`: easier/moderate conditioning, large multi-RHS batches.
- `mixed`: moderate-to-hard conditioning.
- `stress`: hardest among standard presets.
- `a100_heavy`: very large dense configs targeting long 8xA100 runs.

Important table column interpretation:
- `n` in summary table = number of run records, **not problem dimension**.
- `solved` = count with status `solved`.
- `t_med_s` is more robust than `t_mean_s` when there are outliers/failures.
- `g_mean` may be `NaN` for methods whose wrappers do not expose duality gap (e.g., SciPy wrapper).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Update these paths to your run artifacts
combined_results_path = Path('results/comprehensive_<timestamp>/combined_results.jsonl')
manifest_path = Path('results/comprehensive_<timestamp>/manifest.csv')

def load_jsonl(path: Path) -> pd.DataFrame:
    rows = [json.loads(line) for line in path.open() if line.strip()]
    df = pd.json_normalize(rows)
    return df


In [ ]:
df = load_jsonl(combined_results_path)
manifest = pd.read_csv(manifest_path)

for c in ['timing.median_s', 'metrics.primal_residual', 'metrics.dual_residual', 'metrics.duality_gap']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

if 'metadata.comprehensive_case' in df.columns:
    df['case'] = df['metadata.comprehensive_case']
elif 'metadata.overnight_case' in df.columns:
    df['case'] = df['metadata.overnight_case']
else:
    df['case'] = 'unknown'

df['preset'] = df['case'].astype(str).str.split('_s').str[0]

print('rows:', len(df))
print('solvers:', sorted(df['solver'].dropna().unique()))
print('status counts:
', df['status'].value_counts(dropna=False))
display(manifest)


In [ ]:
solver_summary = df.groupby('solver').agg(
    n=('solver', 'size'),
    solved=('status', lambda s: (s == 'solved').sum()),
    t_med_s=('timing.median_s', 'median'),
    t_mean_s=('timing.median_s', 'mean'),
    t_std_s=('timing.median_s', 'std'),
    p_mean=('metrics.primal_residual', 'mean'),
    d_mean=('metrics.dual_residual', 'mean'),
    g_mean=('metrics.duality_gap', 'mean'),
).sort_values('t_med_s')

preset_solver_summary = df.groupby(['preset', 'solver']).agg(
    n=('solver', 'size'),
    solved=('status', lambda s: (s == 'solved').sum()),
    t_med_s=('timing.median_s', 'median'),
    t_mean_s=('timing.median_s', 'mean'),
    p_mean=('metrics.primal_residual', 'mean'),
    d_mean=('metrics.dual_residual', 'mean'),
    g_mean=('metrics.duality_gap', 'mean'),
).reset_index().sort_values(['preset', 'solver'])

display(solver_summary)
display(preset_solver_summary)


In [ ]:
# Representative case: pick first successful stress case if available, else first successful case
ok_cases = manifest.loc[manifest['status'] == 'ok', 'case'].tolist()
rep_case = None
for c in ok_cases:
    if str(c).startswith('stress_'):
        rep_case = c
        break
if rep_case is None and ok_cases:
    rep_case = ok_cases[0]

print('representative case:', rep_case)
rep = df[df['case'] == rep_case].copy()

display(rep[['solver', 'problem_id', 'status', 'timing.median_s', 'metrics.primal_residual', 'metrics.dual_residual', 'metrics.duality_gap']].sort_values(['solver', 'problem_id']))


In [ ]:
# Quick visual for representative case
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for solver, g in rep.groupby('solver'):
    axes[0].scatter(g['timing.median_s'], g['metrics.primal_residual'], label=solver)
    axes[1].scatter(g['timing.median_s'], g['metrics.dual_residual'], label=solver)

for ax, ylab in zip(axes, ['primal residual', 'dual residual']):
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('time (s)')
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=min(4, len(labels)), title='method')
fig.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()


## How to interpret failures / empty folders

If a case folder (e.g., `mixed_s*`, `a100_heavy_s*`) is empty or missing `results.jsonl`, it typically means the case crashed before `reproduce.py` reached final write-out. Common causes:
- OOM or CUDA runtime failure on very large dense batches,
- solver exception in one experiment within that case,
- distributed launch failure (one rank aborts and tears down all ranks).

Use `manifest.csv` first:
- `status=failed` means that case did not complete end-to-end.
- rows in `combined_results.jsonl` therefore represent only successful cases.


## Practical reading checklist

1. Check manifest success/failure by case and preset.
2. Confirm which presets are actually represented in `combined_results.jsonl`.
3. Compare methods first on `t_med_s`, then verify `solved` and residual quality.
4. Treat `t_mean_s` carefully if there are max-iter outliers.
5. For claims, report both aggregate and per-preset results.
